# Stage 1 tuning: joint dilated CNN (validation split only)

This notebook chooses the Stage 1 hyperparameters. It uses **only the train and validation (dev) splits**; the test
split is never loaded into a model here. The chosen configuration is then trained once more in `main.ipynb`, which
is the notebook that holds the required Stage 1 evidence.

**Rules fixed before running:**

1. Grid: learning rate {1e-3, 2e-3} x channels {256, 384} x dropout {0.3, 0.5}, seed 42. Everything else is the
   base configuration shown in Section 6.
2. The two best configurations by dev combined score are re-run with seed 43.
3. The configuration with the higher two-seed mean is chosen.

## 1. Setup

Everything runs from the repository root. Missing data and word vectors are downloaded on first use into
`data/` and `embeddings/` (both git-ignored). On a machine with CUDA the GPU is used; otherwise the CPU.

In [1]:
import os, re, json, time, math, random, copy, zlib, tarfile, zipfile, urllib.request
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

DATA_DIR, EMB_DIR, CKPT_DIR, FIG_DIR = "data", "embeddings", "checkpoints", "figures"
for d in (DATA_DIR, EMB_DIR, CKPT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

MASSIVE_URL = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz"
MASSIVE_PATH = os.path.join(DATA_DIR, "1.0", "data", "en-US.jsonl")
GLOVE_PATH = os.path.join(EMB_DIR, "glove.6B.300d.txt")
GLOVE_URLS = ["https://nlp.stanford.edu/data/glove.6B.zip",
              "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip"]
FASTTEXT_PATH = os.path.join(EMB_DIR, "crawl-300d-2M.vec")
FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip"


def pick_device():
    # On an Apple M2 the CPU was faster than the MPS GPU for these small models (timed: 25 s vs 50+ s per epoch),
    # so MPS is not used. On a machine with CUDA the GPU is used.
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = pick_device()
if "TORCH_THREADS" in os.environ:
    torch.set_num_threads(int(os.environ["TORCH_THREADS"]))


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def download(url, dest):
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, dest)


def ensure_massive():
    if not os.path.exists(MASSIVE_PATH):
        archive = os.path.join(DATA_DIR, "amazon-massive-dataset-1.0.tar.gz")
        download(MASSIVE_URL, archive)
        with tarfile.open(archive) as tar:
            tar.extract("1.0/data/en-US.jsonl", DATA_DIR)
        os.remove(archive)
    return MASSIVE_PATH


def ensure_glove():
    if not os.path.exists(GLOVE_PATH):
        archive = os.path.join(EMB_DIR, "glove.6B.zip")
        for url in GLOVE_URLS:
            try:
                download(url, archive)
                break
            except Exception as err:
                print(f"  failed ({err})")
        with zipfile.ZipFile(archive) as z:
            z.extract("glove.6B.300d.txt", EMB_DIR)
        os.remove(archive)
    return GLOVE_PATH


def ensure_fasttext():
    if not os.path.exists(FASTTEXT_PATH):
        archive = FASTTEXT_PATH + ".zip"
        download(FASTTEXT_URL, archive)
        with zipfile.ZipFile(archive) as z:
            z.extract("crawl-300d-2M.vec", EMB_DIR)
        os.remove(archive)
    return FASTTEXT_PATH


print("torch", torch.__version__, "| device:", DEVICE)

torch 2.12.0 | device: cpu


## 2. Load and verify the dataset

MASSIVE 1.0, `1.0/data/en-US.jsonl`, split with the existing `partition` field. The only model input is `utt`;
labels come from `intent` and `annot_utt`. Nothing is re-split, added or removed.

In [2]:
def load_massive(path):
    with open(path, encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]
    splits = {name: [r for r in records if r["partition"] == name] for name in ("train", "dev", "test")}
    return records, splits


records, raw = load_massive(ensure_massive())
print(f"Total labelled examples: {len(records):,}")
for name in ("train", "dev", "test"):
    print(f"  {name:<5} {len(raw[name]):>6,}")
assert len(records) == 16_521
assert len(raw["train"]) == 11_514 and len(raw["dev"]) == 2_033 and len(raw["test"]) == 2_974
print("Split sizes verified against the brief: 11,514 / 2,033 / 2,974.")
print("Example record:", {k: raw["train"][0][k] for k in ("utt", "intent", "annot_utt")})

Total labelled examples: 16,521
  train 11,514
  dev    2,033
  test   2,974
Split sizes verified against the brief: 11,514 / 2,033 / 2,974.
Example record: {'utt': 'wake me up at nine am on friday', 'intent': 'alarm_set', 'annot_utt': 'wake me up at [time : nine am] on [date : friday]'}


### 2.1 Labels: from `annot_utt` to BIO tags

`annot_utt` marks each slot as `[type : words]`. We turn it into one tag per whitespace token: `B-type` on the first
word of a slot, `I-type` on the following words, `O` elsewhere. The parser asserts that the tokens it recovers are
exactly `utt.split()`, so labels and inputs can never drift apart. The gold set of `(slot type, span text)` pairs is
read straight from `annot_utt` and is what every model is scored against.

In [3]:
SLOT_RE = re.compile(r"\[\s*([^\[\]:]+?)\s*:\s*([^\[\]]*?)\s*\]")


def parse_annot(annot_utt):
    """'wake me up at [time : five am]' -> tokens, BIO tags, gold set of (slot_type, span_text)."""
    tokens, tags, spans, pos = [], [], set(), 0
    for m in SLOT_RE.finditer(annot_utt):
        outside = annot_utt[pos:m.start()].split()
        tokens += outside
        tags += ["O"] * len(outside)
        slot_type, words = m.group(1).strip(), m.group(2).split()
        tokens += words
        tags += [f"B-{slot_type}"] + [f"I-{slot_type}"] * (len(words) - 1)
        spans.add((slot_type, " ".join(words)))
        pos = m.end()
    outside = annot_utt[pos:].split()
    tokens += outside
    tags += ["O"] * len(outside)
    return tokens, tags, spans


def to_example(r):
    tokens, tags, spans = parse_annot(r["annot_utt"])
    assert tokens == r["utt"].split(), f"annot_utt and utt disagree for id {r['id']}"
    return {"id": r["id"], "utt": r["utt"], "tokens": tokens, "tags": tags, "gold_spans": spans,
            "intent": r["intent"], "scenario": r["intent"].split("_", 1)[0]}


data = {name: [to_example(r) for r in raw[name]] for name in ("train", "dev", "test")}
ex = data["train"][0]
print(ex["tokens"], ex["tags"], ex["gold_spans"], ex["intent"], sep="\n")

lengths = [len(e["tokens"]) for e in data["train"]]
print(f"train utterance length: mean {np.mean(lengths):.1f}, 95th pct {np.percentile(lengths, 95):.0f}, max {max(lengths)}")

['wake', 'me', 'up', 'at', 'nine', 'am', 'on', 'friday']
['O', 'O', 'O', 'O', 'B-time', 'I-time', 'O', 'B-date']
{('date', 'friday'), ('time', 'nine am')}
alarm_set
train utterance length: mean 6.9, 95th pct 13, max 35


## 3. Shared evaluation functions

One scorer is used for every experiment in this repository:

* **Intent accuracy** = 100 x correct intents / utterances.
* **Slot micro-F1**: for each utterance, compare the *set* of predicted `(slot type, span text)` pairs with the gold
  set. A true positive needs both the type and the exact span. TP, FP and FN are pooled over all utterances, and
  F1 = 100 x 2TP / (2TP + FP + FN).
* Turning tags into spans is part of the model, not the scorer. An `I-x` that does not continue an `x` span starts a
  new span (the usual CoNLL convention), so an invalid tag sequence is still scored, never silently repaired.

The checks below test hand-built cases, and confirm that gold tags reproduce 100.00 on every split.

In [4]:
def tags_to_spans(tokens, tags):
    """BIO tags -> set of (slot_type, span_text). An I-x that does not continue an x span starts a new span."""
    spans, cur_type, cur_words = set(), None, []
    for tok, tag in zip(tokens, tags):
        if tag == "O":
            if cur_type:
                spans.add((cur_type, " ".join(cur_words)))
            cur_type, cur_words = None, []
        elif tag.startswith("B-") or tag[2:] != cur_type:
            if cur_type:
                spans.add((cur_type, " ".join(cur_words)))
            cur_type, cur_words = tag[2:], [tok]
        else:
            cur_words.append(tok)
    if cur_type:
        spans.add((cur_type, " ".join(cur_words)))
    return spans


def intent_accuracy(pred_intents, gold_intents):
    return 100.0 * sum(p == g for p, g in zip(pred_intents, gold_intents)) / len(gold_intents)


def slot_counts(pred_span_sets, gold_span_sets):
    tp = fp = fn = 0
    for pred, gold in zip(pred_span_sets, gold_span_sets):
        tp += len(pred & gold)
        fp += len(pred - gold)
        fn += len(gold - pred)
    return tp, fp, fn


def slot_micro_f1(pred_span_sets, gold_span_sets):
    tp, fp, fn = slot_counts(pred_span_sets, gold_span_sets)
    return 100.0 * 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 100.0


def score(pred_intents, pred_spans, examples):
    """The one scoring rule used for every experiment in this project."""
    gold_intents = [e["intent"] for e in examples]
    gold_spans = [e["gold_spans"] for e in examples]
    tp, fp, fn = slot_counts(pred_spans, gold_spans)
    acc = intent_accuracy(pred_intents, gold_intents)
    f1 = slot_micro_f1(pred_spans, gold_spans)
    return {"intent_acc": acc, "slot_f1": f1, "combined": (acc + f1) / 2,
            "intent_correct": sum(p == g for p, g in zip(pred_intents, gold_intents)), "tp": tp, "fp": fp, "fn": fn}


# Unit tests on hand-built cases
toks = "wake me up at five am this week".split()
assert tags_to_spans(toks, ["O", "O", "O", "O", "B-time", "I-time", "B-date", "I-date"]) == {("time", "five am"), ("date", "this week")}
assert tags_to_spans(toks, ["O", "O", "O", "O", "I-time", "I-time", "O", "O"]) == {("time", "five am")}
assert tags_to_spans(toks, ["O", "O", "O", "O", "B-time", "I-date", "O", "O"]) == {("time", "five"), ("date", "am")}
assert slot_counts([{("a", "x"), ("b", "y")}], [{("a", "x"), ("b", "z")}]) == (1, 1, 1)
assert abs(slot_micro_f1([{("a", "x"), ("b", "y")}], [{("a", "x"), ("b", "z")}]) - 50.0) < 1e-9
assert intent_accuracy(["a", "b", "c", "d"], ["a", "b", "x", "d"]) == 75.0
# Gold tags must reproduce the gold span sets exactly on every split
for name, exs in data.items():
    s = score([e["intent"] for e in exs], [tags_to_spans(e["tokens"], e["tags"]) for e in exs], exs)
    assert s["intent_acc"] == 100.0 and s["slot_f1"] == 100.0, name
    print(f"{name}: gold round trip -> intent {s['intent_acc']:.2f}, slot F1 {s['slot_f1']:.2f}, gold slots {s['tp']:,}")

train: gold round trip -> intent 100.00, slot F1 100.00, gold slots 11,340
dev: gold round trip -> intent 100.00, slot F1 100.00, gold slots 2,011


test: gold round trip -> intent 100.00, slot F1 100.00, gold slots 2,815


**Test lock.** The test split can only be scored after a stage's checkpoint is frozen. `check_split_allowed` raises
an error otherwise, which makes accidental test-set tuning impossible rather than just discouraged.

In [5]:
TEST_UNLOCKED = {"stage1": False, "stage2": False}


def check_split_allowed(split, stage):
    if split == "test" and not TEST_UNLOCKED[stage]:
        raise RuntimeError(f"Test split is locked for {stage}: freeze the checkpoint first.")

## 4. Vocabularies, word vectors and batching

Intent, scenario, tag, word and character vocabularies are built from the **training split only**. Dev and test words
never seen in training map to `<unk>` (the character-level features still see their spelling).

Word vectors (GloVe, fastText) are the only pre-trained component, as the brief allows. `vector_keys` tries the exact
word, then simple normalisations (`p.m.` -> `pm`), then the average of the parts of a contraction or hyphenated word.

In [6]:
PAD, UNK = "<pad>", "<unk>"


class Vocab:
    def __init__(self, items, specials=(PAD, UNK)):
        self.itos = list(specials) + [x for x in items if x not in specials]
        self.stoi = {x: i for i, x in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def get(self, x):
        return self.stoi.get(x, self.stoi.get(UNK, 0))


def by_frequency(counter):
    return [w for w, _ in sorted(counter.items(), key=lambda kv: (-kv[1], kv[0]))]


intent_vocab = Vocab(sorted({e["intent"] for e in data["train"]}), specials=())
scenario_vocab = Vocab(sorted({e["scenario"] for e in data["train"]}), specials=())
slot_types = sorted({t[2:] for e in data["train"] for t in e["tags"] if t != "O"})
tag_vocab = Vocab(["O"] + [f"{p}-{s}" for s in slot_types for p in ("B", "I")], specials=())
intent_to_scenario = torch.tensor([scenario_vocab.stoi[i.split("_", 1)[0]] for i in intent_vocab.itos])
for name in ("dev", "test"):
    assert {e["intent"] for e in data[name]} <= set(intent_vocab.stoi), name
    assert {t for e in data[name] for t in e["tags"]} <= set(tag_vocab.stoi), name
print(f"intents {len(intent_vocab)} | scenarios {len(scenario_vocab)} | slot types {len(slot_types)} | BIO tags {len(tag_vocab)}")

intents 60 | scenarios 18 | slot types 55 | BIO tags 111


In [7]:
def vector_keys(w):
    """GloVe/fastText keys to try for a token, in order. A tuple means 'average these vectors'."""
    s = w.strip(".,!?;:\"()")
    keys = [w, s, s.replace(".", "")]
    m = re.fullmatch(r"(.+?)(n't|'s|'re|'ve|'ll|'d|'m|')", s)
    if m:
        keys.append(tuple(p for p in m.groups() if p))
    parts = [p for p in re.split(r"[-./&]", s) if p]
    if len(parts) > 1:
        keys.append(tuple(parts))
    return keys


def read_vectors(path, wanted, top_k=0):
    """Stream a .txt/.vec file. Keep rows whose word is in `wanted`, plus the first `top_k` rows (most frequent)."""
    vecs, top = {}, []
    with open(path, encoding="utf-8", errors="ignore") as f:
        for rank, line in enumerate(f):
            word, _, rest = line.rstrip("\n").partition(" ")
            if rank == 0 and rest.count(" ") < 5:      # fastText header line "2000000 300"
                continue
            if word in wanted or len(top) < top_k:
                v = np.asarray(rest.split(" "), dtype=np.float32)
                if v.shape[0] != 300:
                    continue
                vecs[word] = v
                if len(top) < top_k:
                    top.append(word)
    return vecs, top


def lookup_vector(w, vecs):
    for k in vector_keys(w):
        if isinstance(k, tuple):
            if all(p in vecs for p in k):
                return np.mean([vecs[p] for p in k], axis=0)
        elif k in vecs:
            return vecs[k]
    return None


def needed_keys(words):
    return {p for w in words for k in vector_keys(w) for p in (k if isinstance(k, tuple) else (k,))}

In [8]:
def pad2d(rows, pad=0):
    width = max(1, max(len(r) for r in rows))
    out = torch.full((len(rows), width), pad, dtype=torch.long)
    for i, r in enumerate(rows):
        out[i, :len(r)] = torch.tensor(r, dtype=torch.long)
    return out


def pad3d(rows, pad=0):
    d1 = max(1, max(len(r) for r in rows))
    d2 = max(1, max((len(c) for r in rows for c in r), default=1))
    out = torch.full((len(rows), d1, d2), pad, dtype=torch.long)
    for i, r in enumerate(rows):
        for j, c in enumerate(r):
            out[i, j, :len(c)] = torch.tensor(c, dtype=torch.long)
    return out


def collate(items):
    batch = {"lengths": torch.tensor([len(it["tags"]) for it in items]),
             "unit_lengths": torch.tensor([len(it["units"]) for it in items]),
             "tags": pad2d([it["tags"] for it in items]),
             "first": pad2d([it["first"] for it in items]),
             "intent": torch.tensor([it["intent"] for it in items]),
             "scenario": torch.tensor([it["scenario"] for it in items]),
             "units": pad2d([it["units"] for it in items]),
             "chars": pad3d([it["chars"] for it in items])}
    if "frozen" in items[0]:
        batch["frozen"] = pad2d([it["frozen"] for it in items])
    if "grams" in items[0]:
        batch["grams"] = pad3d([it["grams"] for it in items])
    return batch


def make_batches(encoded, batch_size, shuffle, seed=0):
    """Evaluation keeps dataset order. Training shuffles, then sorts by length inside pools of 50 batches
    (less padding, so faster), then shuffles the batch order."""
    order = list(range(len(encoded)))
    if shuffle:
        rng = random.Random(seed)
        rng.shuffle(order)
        pool = batch_size * 50
        order = [i for s in range(0, len(order), pool)
                 for i in sorted(order[s:s + pool], key=lambda i: len(encoded[i]["units"]))]
    batches = [order[s:s + batch_size] for s in range(0, len(order), batch_size)]
    if shuffle:
        rng.shuffle(batches)
    return [collate([encoded[i] for i in b]) for b in batches]


def to_device(batch):
    return {k: (v if k in ("lengths", "unit_lengths") else v.to(DEVICE)) for k, v in batch.items()}

## 5. Linear-chain CRF (shared by both stages)

A per-token softmax picks each tag on its own, so it can output `O I-time`, which is not a valid span. A
linear-chain CRF scores the **whole tag sequence**: the sum of per-token emission scores plus a learned score for
every tag-to-tag transition. Training maximises the log-probability of the gold sequence (the forward algorithm sums
over all sequences); prediction uses Viterbi, the same dynamic programme as in HMM tagging, to find the best
sequence exactly. The implementation is checked against brute-force enumeration of every possible path.

In [9]:
def crf_log_partition(em, mask, trans, start, end):
    alpha = start + em[:, 0]
    for t in range(1, em.size(1)):
        nxt = torch.logsumexp(alpha.unsqueeze(2) + trans.unsqueeze(0), dim=1) + em[:, t]
        alpha = torch.where(mask[:, t].unsqueeze(1), nxt, alpha)
    return torch.logsumexp(alpha + end, dim=1)


def crf_path_score(em, tags, mask, trans, start, end):
    maskf = mask.float()
    s = start[tags[:, 0]] + (em.gather(2, tags.unsqueeze(2)).squeeze(2) * maskf).sum(1)
    s = s + (trans[tags[:, :-1], tags[:, 1:]] * maskf[:, 1:]).sum(1)
    last = tags.gather(1, (mask.sum(1) - 1).unsqueeze(1)).squeeze(1)
    return s + end[last]


def crf_viterbi(em, mask, trans, start, end):
    """Best tag path per sequence and its score."""
    score, history = start + em[:, 0], []
    for t in range(1, em.size(1)):
        best, idx = (score.unsqueeze(2) + trans.unsqueeze(0)).max(dim=1)
        score = torch.where(mask[:, t].unsqueeze(1), best + em[:, t], score)
        history.append(idx)
    score = score + end
    best_score, last = score.max(dim=1)
    hist = torch.stack(history, 1).cpu() if history else None
    last, lengths = last.cpu().tolist(), mask.sum(1).cpu().tolist()
    paths = []
    for b, L in enumerate(lengths):
        path = [last[b]]
        for t in range(L - 1, 0, -1):
            path.append(int(hist[b, t - 1, path[-1]]))
        paths.append(path[::-1])
    return paths, best_score


def bio_constraints(tag_itos):
    """allowed[i, j]: may tag j follow tag i? An I-x may only follow B-x or I-x, and may not start a sequence."""
    n = len(tag_itos)
    allowed, start_ok = torch.ones(n, n, dtype=torch.bool), torch.ones(n, dtype=torch.bool)
    for j, tj in enumerate(tag_itos):
        if tj.startswith("I-"):
            start_ok[j] = False
            for i, ti in enumerate(tag_itos):
                allowed[i, j] = ti[2:] == tj[2:] and ti != "O"
    return allowed, start_ok


class CRF(nn.Module):
    """Linear-chain CRF over BIO tags. With constraints, illegal transitions get a -10,000 penalty."""

    def __init__(self, n_tags, constraints=None):
        super().__init__()
        self.trans = nn.Parameter(torch.zeros(n_tags, n_tags))
        self.start = nn.Parameter(torch.zeros(n_tags))
        self.end = nn.Parameter(torch.zeros(n_tags))
        pen_t, pen_s = torch.zeros(n_tags, n_tags), torch.zeros(n_tags)
        if constraints is not None:
            allowed, start_ok = constraints
            pen_t, pen_s = (~allowed).float() * -1e4, (~start_ok).float() * -1e4
        self.register_buffer("pen_trans", pen_t)
        self.register_buffer("pen_start", pen_s)

    def potentials(self):
        return self.trans + self.pen_trans, self.start + self.pen_start, self.end

    def nll(self, em, tags, mask):
        tr, st, en = self.potentials()
        return crf_log_partition(em, mask, tr, st, en) - crf_path_score(em, tags, mask, tr, st, en)


def _check_crf_brute_force(n_tags=4, T=4, trials=5):
    import itertools
    g = torch.Generator().manual_seed(0)
    for _ in range(trials):
        em = torch.randn(2, T, n_tags, generator=g)
        tr, st, en = (torch.randn(n_tags, n_tags, generator=g), torch.randn(n_tags, generator=g), torch.randn(n_tags, generator=g))
        mask = torch.tensor([[1] * T, [1] * (T - 1) + [0]], dtype=torch.bool)
        logz = crf_log_partition(em, mask, tr, st, en)
        paths, best = crf_viterbi(em, mask, tr, st, en)
        for b in range(2):
            L = int(mask[b].sum())
            scores = {}
            for p in itertools.product(range(n_tags), repeat=L):
                tags = torch.tensor([list(p) + [0] * (T - L)])
                scores[p] = float(crf_path_score(em[b:b + 1], tags, mask[b:b + 1], tr, st, en))
            assert abs(float(logz[b]) - float(torch.logsumexp(torch.tensor(list(scores.values())), 0))) < 1e-4
            assert tuple(paths[b]) == max(scores, key=scores.get) and abs(float(best[b]) - max(scores.values())) < 1e-4
    return True


assert _check_crf_brute_force()
print("CRF forward algorithm and Viterbi match brute-force enumeration.")

CRF forward algorithm and Viterbi match brute-force enumeration.


### 5.1 Training loop and checkpoint selection

`fit` trains on the train split only. After each epoch it scores dev with the shared scorer, logs the train and dev
losses, dev scores, learning rate and time, and saves a checkpoint whenever the **dev combined score**
((intent accuracy + slot F1) / 2) improves. Combined is used because the brief weights both tasks and a model that
trades one for the other should not be rewarded.

In [10]:
def count_params(model):
    params = list(model.parameters())          # nn.Module.parameters() already de-duplicates shared tensors
    buffers = [b for n, b in model.named_buffers() if n.endswith("knn_keys")]
    total = sum(p.numel() for p in params) + sum(b.numel() for b in buffers)
    trainable = sum(p.numel() for p in params if p.requires_grad)
    return total, trainable


@torch.no_grad()
def predict(model, batches, examples):
    """Average loss, predicted intents and predicted span sets, in dataset order."""
    model.eval()
    total_loss, n, intents, spans, idx = 0.0, 0, [], [], 0
    for batch in batches:
        batch = to_device(batch)
        out = model.compute(batch)
        total_loss += model.loss(batch, out).item() * len(batch["lengths"])
        n += len(batch["lengths"])
        intent_ids, tag_paths = model.decode(batch, out)
        for i, path in zip(intent_ids, tag_paths):
            ex = examples[idx]
            intents.append(intent_vocab.itos[i])
            spans.append(tags_to_spans(ex["tokens"], [tag_vocab.itos[t] for t in path]))
            idx += 1
    return total_loss / n, intents, spans


def evaluate(model, batches, examples):
    loss, intents, spans = predict(model, batches, examples)
    return {"loss": loss, **score(intents, spans, examples)}


def fit(model, cfg, train_enc, dev_batches, dev_examples, ckpt_path, verbose=True):
    """Train on the train split only; after every epoch score dev and keep the best dev-combined checkpoint.
    With cfg['swa_start'], the running average of the weights from that epoch on is also scored as a candidate."""
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    steps_per_epoch = math.ceil(len(train_enc) / cfg["batch_size"])
    total_steps = steps_per_epoch * cfg["max_epochs"]
    if cfg["schedule"] == "cosine":
        warm = steps_per_epoch * cfg.get("warmup_epochs", 1)
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total_steps))))
    else:
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=cfg["plateau_patience"])
    swa, swa_n = None, 0
    history, best, bad = [], {"combined": -1.0}, 0
    t_start = time.time()
    for epoch in range(1, cfg["max_epochs"] + 1):
        t0 = time.time()
        model.train()
        running, seen = 0.0, 0
        for batch in make_batches(train_enc, cfg["batch_size"], shuffle=True, seed=cfg["seed"] * 1000 + epoch):
            batch = to_device(batch)
            loss = model.loss(batch, model.compute(batch))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(params, cfg["clip"])
            opt.step()
            if cfg["schedule"] == "cosine":
                sched.step()
            running += loss.item() * len(batch["lengths"])
            seen += len(batch["lengths"])
        dev = evaluate(model, dev_batches, dev_examples)
        if cfg["schedule"] != "cosine":
            sched.step(dev["combined"])
        candidates = [("raw", dev, None)]
        if cfg.get("swa_start") and epoch >= cfg["swa_start"]:
            state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            if swa is None:
                swa = {k: v.clone() for k, v in state.items()}
            else:
                for k, v in state.items():
                    if v.dtype.is_floating_point:
                        swa[k].mul_(swa_n / (swa_n + 1)).add_(v, alpha=1 / (swa_n + 1))
            swa_n += 1
            model.load_state_dict(swa)
            candidates.append(("swa", evaluate(model, dev_batches, dev_examples), copy.deepcopy(swa)))
            model.load_state_dict(state)
        row = {"epoch": epoch, "train_loss": running / seen, "dev_loss": dev["loss"], "dev_intent_acc": dev["intent_acc"],
               "dev_slot_f1": dev["slot_f1"], "dev_combined": dev["combined"], "lr": opt.param_groups[0]["lr"],
               "epoch_time_s": time.time() - t0}
        if len(candidates) > 1:
            s = candidates[1][1]
            row.update({"swa_intent_acc": s["intent_acc"], "swa_slot_f1": s["slot_f1"], "swa_combined": s["combined"]})
        history.append(row)
        improved = ""
        for kind, res, state in candidates:
            if res["combined"] > best["combined"]:
                best = {**res, "epoch": epoch, "kind": kind}
                torch.save({"state": state if state is not None else model.state_dict(), "epoch": epoch, "kind": kind,
                            "dev": res, "config": cfg}, ckpt_path)
                improved = f"  <- saved ({kind})"
                bad = -1
        bad += 1
        if verbose:
            extra = f" | swa comb {row['swa_combined']:.2f}" if "swa_combined" in row else ""
            print(f"epoch {epoch:2d} | train loss {row['train_loss']:.4f} | dev loss {row['dev_loss']:.4f} | "
                  f"intent {row['dev_intent_acc']:.2f} | slot F1 {row['dev_slot_f1']:.2f} | combined {row['dev_combined']:.2f}"
                  f"{extra} | {row['epoch_time_s']:.1f}s{improved}")
        if bad >= cfg["early_stop"]:
            if verbose:
                print(f"Early stop: no dev improvement for {cfg['early_stop']} epochs.")
            break
    best["train_time_s"] = time.time() - t_start
    return history, best


def plot_history(history, best_epoch, title, path):
    ep = [h["epoch"] for h in history]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    ax[0].plot(ep, [h["train_loss"] for h in history], label="train loss")
    ax[0].plot(ep, [h["dev_loss"] for h in history], label="validation loss")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].set_title(f"{title}: loss"); ax[0].legend()
    ax[1].plot(ep, [h["dev_intent_acc"] for h in history], label="val intent accuracy")
    ax[1].plot(ep, [h["dev_slot_f1"] for h in history], label="val slot micro-F1")
    if "swa_combined" in history[-1]:
        sw = [h for h in history if "swa_combined" in h]
        ax[1].plot([h["epoch"] for h in sw], [h["swa_intent_acc"] for h in sw], "--", label="SWA intent")
        ax[1].plot([h["epoch"] for h in sw], [h["swa_slot_f1"] for h in sw], "--", label="SWA slot F1")
    ax[1].axvline(best_epoch, color="grey", ls=":", label=f"selected epoch {best_epoch}")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("%"); ax[1].set_title(f"{title}: validation scores"); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

## 6. Stage 1 model and inputs

The model is described in full in `main.ipynb` Section 6. In short: GloVe-initialised word embeddings and a
character CNN feed an iterated dilated convolution block; the intent head pools over the sentence, and the slot
head feeds a CRF.

In [11]:
def whitespace_units(token):
    return [token]


def build_stage1_features(train_examples, glove_vecs, word_drop_alpha):
    """Stage 1 inputs: one unit per whitespace token. Word vocabulary and characters come from train only."""
    counts = Counter(t for e in train_examples for t in e["tokens"])
    words = Vocab(by_frequency(counts))
    chars = Vocab(sorted({c for e in train_examples for t in e["tokens"] for c in t}))
    rng = np.random.RandomState(0)
    emb, found = rng.normal(0, 0.3, (len(words), 300)).astype(np.float32), 0
    emb[0] = 0
    for i, w in enumerate(words.itos[2:], start=2):
        v = lookup_vector(w, glove_vecs)
        if v is not None:
            emb[i], found = v, found + 1
    drop_p = torch.zeros(len(words))
    for i, w in enumerate(words.itos[2:], start=2):
        drop_p[i] = word_drop_alpha / (word_drop_alpha + counts[w])
    return {"words": words, "chars": chars, "emb": torch.tensor(emb), "drop_p": drop_p,
            "coverage": found / (len(words) - 2)}


def encode_stage1(examples, feats, max_chars=20):
    out = []
    for e in examples:
        out.append({"units": [feats["words"].get(t) for t in e["tokens"]],
                    "chars": [[feats["chars"].get(c) for c in t[:max_chars]] for t in e["tokens"]],
                    "first": list(range(len(e["tokens"]))),
                    "tags": [tag_vocab.stoi[t] for t in e["tags"]],
                    "intent": intent_vocab.stoi[e["intent"]], "scenario": scenario_vocab.stoi[e["scenario"]]})
    return out

In [12]:
class SeqConv(nn.Module):
    """Width-k 1-D convolution with dilation d over (B, T, C) inputs, zero padded to keep length T.
    Written as k shifted copies of the input plus one linear layer: the same function as nn.Conv1d,
    but it avoids PyTorch's slow dilated-convolution path on CPU and Apple GPUs."""

    def __init__(self, c_in, c_out, width=3, dilation=1):
        super().__init__()
        assert width % 2 == 1
        self.width, self.dilation = width, dilation
        self.lin = nn.Linear(width * c_in, c_out)

    def forward(self, x):
        T, pad = x.size(1), (self.width // 2) * self.dilation
        xp = F.pad(x, (0, 0, pad, pad))
        return self.lin(torch.cat([xp[:, i * self.dilation:i * self.dilation + T] for i in range(self.width)], dim=2))


class CharCNN(nn.Module):
    """Characters of each word -> convolution -> max over positions: one vector per word, also for unseen words."""

    def __init__(self, n_chars, char_dim=50, filters=100, width=3):
        super().__init__()
        self.emb = nn.Embedding(n_chars, char_dim, padding_idx=0)
        self.conv = SeqConv(char_dim, filters, width)
        self.out_dim = filters

    def forward(self, chars):                                   # (B, T, C)
        B, T, C = chars.shape
        flat = chars.view(B * T, C)
        h = self.conv(self.emb(flat)).masked_fill((flat == 0).unsqueeze(2), -1e4).max(dim=1).values
        return torch.tanh(h).view(B, T, -1)


class DilatedBlock(nn.Module):
    """Three width-3 convolutions with dilation 1, 2, 4. Each has a residual connection and layer norm.
    Padding positions are zeroed after every layer so they never leak into real tokens."""

    def __init__(self, channels, dilations=(1, 2, 4), width=3, dropout=0.3):
        super().__init__()
        self.convs = nn.ModuleList([SeqConv(channels, channels, width, d) for d in dilations])
        self.norms = nn.ModuleList([nn.LayerNorm(channels) for _ in dilations])
        self.drop = nn.Dropout(dropout)

    def forward(self, x, m):                                    # x (B, T, C), m (B, T, 1)
        for conv, norm in zip(self.convs, self.norms):
            h = F.relu(conv(x))
            x = norm(x + self.drop(h)) * m
        return x


def masked_mean_max(h, mask):
    m = mask.unsqueeze(2).float()
    mean = (h * m).sum(1) / m.sum(1).clamp(min=1)
    mx = h.masked_fill(m == 0, -1e4).max(1).values
    return torch.cat([mean, mx], dim=1)


class Stage1IDCNN(nn.Module):
    """Joint ID-CNN: [GloVe-initialised word embedding ; char-CNN] -> iterated dilated conv block (shared weights)
    -> intent from mean+max pooling, slots from a CRF over per-token emissions."""

    def __init__(self, cfg, feats):
        super().__init__()
        self.cfg, C = cfg, cfg["channels"]
        self.word_emb = nn.Embedding.from_pretrained(feats["emb"].clone(), freeze=False, padding_idx=0)
        self.char_cnn = CharCNN(len(feats["chars"]))
        self.in_drop = nn.Dropout(cfg["dropout"])
        self.proj = nn.Linear(300 + self.char_cnn.out_dim, C)
        self.block = DilatedBlock(C, cfg["dilations"], dropout=cfg["dropout"])
        self.intent_head = nn.Sequential(nn.Linear(2 * C, C), nn.ReLU(), nn.Dropout(cfg["dropout"]),
                                         nn.Linear(C, len(intent_vocab)))
        self.slot_drop = nn.Dropout(cfg["dropout"])
        self.slot_out = nn.Linear(C, len(tag_vocab))
        self.crf = CRF(len(tag_vocab))
        self.register_buffer("drop_p", feats["drop_p"])

    def compute(self, batch):
        ids, mask = batch["units"], batch["units"] != 0
        if self.training:                                       # rare words are sometimes replaced by <unk>
            ids = ids.masked_fill((torch.rand(ids.shape, device=ids.device) < self.drop_p[ids]) & mask, 1)
        m = mask.unsqueeze(2).float()
        x = torch.cat([self.word_emb(ids), self.char_cnn(batch["chars"])], dim=2)
        x = self.proj(self.in_drop(x)) * m
        for _ in range(self.cfg["iterations"]):
            x = self.block(x, m)
        intent_logits = self.intent_head(masked_mean_max(x, mask))
        return {"intent_logits": intent_logits, "emissions": self.slot_out(self.slot_drop(x)), "mask": mask}

    def loss(self, batch, out):
        nll = self.crf.nll(out["emissions"], batch["tags"], out["mask"]).mean()
        return F.cross_entropy(out["intent_logits"], batch["intent"]) + self.cfg["slot_weight"] * nll

    def decode(self, batch, out):
        paths, _ = crf_viterbi(out["emissions"], out["mask"], *self.crf.potentials())
        return out["intent_logits"].argmax(1).tolist(), paths

In [13]:
s1_words = {t for e in data["train"] for t in e["tokens"]}
glove_s1, _ = read_vectors(ensure_glove(), needed_keys(s1_words))
S1_BASE = dict(channels=256, dilations=(1, 2, 4), iterations=2, dropout=0.3, lr=1e-3, weight_decay=0.01,
               batch_size=32, max_epochs=50, schedule="plateau", plateau_patience=3, early_stop=10, clip=5.0,
               slot_weight=1.0, word_drop_alpha=0.25, seed=42)
feats1 = build_stage1_features(data["train"], glove_s1, S1_BASE["word_drop_alpha"])
train1 = encode_stage1(data["train"], feats1)
dev1_batches = make_batches(encode_stage1(data["dev"], feats1), 256, shuffle=False)
print(f"Stage 1 word vocabulary (train only): {len(feats1['words']):,} | characters: {len(feats1['chars'])} | "
      f"train words with a GloVe vector: {100 * feats1['coverage']:.2f}%")

Stage 1 word vocabulary (train only): 5,304 | characters: 34 | train words with a GloVe vector: 96.55%


## 7. Grid search (seed 42)

In [14]:
def run_stage1(over, seed, verbose=False):
    cfg = {**S1_BASE, **over, "seed": seed}
    set_seed(seed)
    model = Stage1IDCNN(cfg, feats1).to(DEVICE)
    hist, best = fit(model, cfg, train1, dev1_batches, data["dev"], os.path.join(CKPT_DIR, "tuning_scratch.pt"), verbose)
    return {**over, "seed": seed, "intent_acc": best["intent_acc"], "slot_f1": best["slot_f1"],
            "combined": best["combined"], "best_epoch": best["epoch"], "epochs_run": len(hist),
            "minutes": best["train_time_s"] / 60, "params": count_params(model)[0]}


GRID = [{"lr": lr, "channels": c, "dropout": d} for lr in (1e-3, 2e-3) for c in (256, 384) for d in (0.3, 0.5)]
grid_results = []
for i, over in enumerate(GRID, 1):
    r = run_stage1(over, seed=42)
    grid_results.append(r)
    print(f"[{i}/{len(GRID)}] lr {r['lr']:.0e} | channels {r['channels']} | dropout {r['dropout']} -> dev intent "
          f"{r['intent_acc']:.2f} | slot F1 {r['slot_f1']:.2f} | combined {r['combined']:.2f} | best epoch "
          f"{r['best_epoch']}/{r['epochs_run']} | {r['minutes']:.1f} min")
os.makedirs("results", exist_ok=True)
json.dump(grid_results, open("results/stage1_grid.json", "w"), indent=1)

[1/8] lr 1e-03 | channels 256 | dropout 0.3 -> dev intent 86.42 | slot F1 82.16 | combined 84.29 | best epoch 30/40 | 24.3 min


[2/8] lr 1e-03 | channels 256 | dropout 0.5 -> dev intent 86.87 | slot F1 81.79 | combined 84.33 | best epoch 42/50 | 28.3 min


[3/8] lr 1e-03 | channels 384 | dropout 0.3 -> dev intent 87.01 | slot F1 82.64 | combined 84.83 | best epoch 45/50 | 40.1 min


[4/8] lr 1e-03 | channels 384 | dropout 0.5 -> dev intent 86.97 | slot F1 82.08 | combined 84.52 | best epoch 40/50 | 28.2 min


[5/8] lr 2e-03 | channels 256 | dropout 0.3 -> dev intent 86.08 | slot F1 81.47 | combined 83.78 | best epoch 41/50 | 25.7 min


[6/8] lr 2e-03 | channels 256 | dropout 0.5 -> dev intent 85.24 | slot F1 79.77 | combined 82.51 | best epoch 24/34 | 9.5 min


[7/8] lr 2e-03 | channels 384 | dropout 0.3 -> dev intent 85.78 | slot F1 81.21 | combined 83.49 | best epoch 38/48 | 11.1 min


[8/8] lr 2e-03 | channels 384 | dropout 0.5 -> dev intent 82.64 | slot F1 79.83 | combined 81.24 | best epoch 49/50 | 11.3 min


## 8. Seed check of the two best configurations

In [15]:
ranked = sorted(grid_results, key=lambda r: -r["combined"])
seed_results = []
for r in ranked[:2]:
    over = {k: r[k] for k in ("lr", "channels", "dropout")}
    r2 = run_stage1(over, seed=43)
    seed_results.append({"config": over, "seed42": r["combined"], "seed43": r2["combined"],
                         "mean": (r["combined"] + r2["combined"]) / 2, "seed43_detail": r2})
    print(f"{over} -> combined seed 42 {r['combined']:.2f} | seed 43 {r2['combined']:.2f} | "
          f"mean {(r['combined'] + r2['combined']) / 2:.2f}")
chosen = max(seed_results, key=lambda s: s["mean"])
print("Chosen Stage 1 configuration (highest two-seed mean dev combined):", chosen["config"])
json.dump({"seed_check": seed_results, "chosen": chosen["config"]}, open("results/stage1_seed_check.json", "w"), indent=1)

{'lr': 0.001, 'channels': 384, 'dropout': 0.3} -> combined seed 42 84.83 | seed 43 83.93 | mean 84.38


{'lr': 0.001, 'channels': 384, 'dropout': 0.5} -> combined seed 42 84.52 | seed 43 84.48 | mean 84.50
Chosen Stage 1 configuration (highest two-seed mean dev combined): {'lr': 0.001, 'channels': 384, 'dropout': 0.5}
